# IMC25 scoring metrics (clustering)

This notebook computes the **clustering** part of the IMC25 evaluation described in the competition rules.

For a ground-truth scene $S$ and a predicted cluster $C$:

- **mAA (recall-like):**  $\mathrm{mAA}(S,C)=\frac{|S\cap C|}{|S|}$
- **Clustering score (precision-like):**  $\mathrm{CS}(S,C)=\frac{|S\cap C|}{|C|}$

Each ground-truth scene $S$ is assigned to the predicted cluster $C_{\phi(S)}$ that **maximizes mAA** (ties broken by higher clustering score). Predicted **outliers** are excluded from this assignment.

Dataset aggregation (micro-average across images):

- $\mathrm{mAA}=\frac{\sum_S |S\cap C_{\phi(S)}|}{\sum_S |S|}$
- $\mathrm{CS}=\frac{\sum_S |S\cap C_{\phi(S)}|}{\sum_S |C_{\phi(S)}|}$

Final score (harmonic mean):

$$\mathrm{Final}=\frac{2}{\frac{1}{\mathrm{mAA}}+\frac{1}{\mathrm{CS}}}$$

> Note: this notebook evaluates **scene clustering** only. For pose-aware mAA (thresholded camera-center accuracy), see `notebooks/08_pose_scoring_metrics.ipynb`.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

from IPython.display import display
import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


def _image_keys_in_dir(dataset_dir: Path) -> set[str]:
    if not dataset_dir.exists():
        return set()
    keys: set[str] = set()
    for p in dataset_dir.iterdir():
        if not p.is_file():
            continue
        if p.suffix.lower() not in IMAGE_EXTS:
            continue
        keys.add(p.stem)
    return keys


def normalize_image_key(dataset: str, *, image_id: Optional[str] = None, image: Optional[str] = None) -> str:
    """Normalize various ID formats to a stable image key (filename stem)."""
    if isinstance(image, str) and image.strip() and image.lower() != "nan":
        return Path(image).stem

    if not isinstance(image_id, str) or not image_id.strip():
        raise ValueError("Need either `image` or `image_id` to derive an image key")

    s = image_id.strip()
    # Common submission IDs: "{dataset}_{image}.png_public" → "{image}.png"
    if s.endswith("_public") or s.endswith("_private"):
        s = s.rsplit("_", 1)[0]
    prefix = f"{dataset}_"
    if s.startswith(prefix):
        s = s[len(prefix) :]

    return Path(s).stem


def load_ground_truth(
    *,
    labels_csv: Path = Path("data/train_labels.csv"),
    data_root: Path = Path("data/train"),
) -> Tuple[Dict[str, Dict[str, set[str]]], Dict[str, set[str]]]:
    """Return ground-truth scene→images and all-images-per-dataset (to infer GT outliers)."""
    gt = pd.read_csv(labels_csv)
    required = {"dataset", "scene", "image"}
    missing = required - set(gt.columns)
    if missing:
        raise ValueError(f"Ground truth missing columns: {sorted(missing)}")

    gt = gt[["dataset", "scene", "image"]].copy()
    gt["image_key"] = gt["image"].map(lambda x: Path(str(x)).stem)

    scenes: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, scene), g in gt.groupby(["dataset", "scene"], sort=True):
        scenes.setdefault(dataset, {})[scene] = set(g["image_key"].tolist())

    all_images: Dict[str, set[str]] = {}
    for dataset in sorted(scenes.keys()):
        keys = _image_keys_in_dir(data_root / dataset)
        # Fall back to labeled images only if the folder is missing.
        if not keys:
            keys = set(gt.loc[gt["dataset"] == dataset, "image_key"].tolist())
        all_images[dataset] = keys

    return scenes, all_images


def load_predictions(
    *,
    pred_csv: Path,
    expected_images: Dict[str, set[str]],
    outliers_label: str = "outliers",
) -> Dict[str, Dict[str, set[str]]]:
    """Load predicted cluster labels. Returns dataset→cluster→image_keys.

    Supports typical formats:
    - cache/clusters.csv: dataset,image_id,scene
    - submission.csv: dataset,scene,image,rotation_matrix,translation_vector
    - richer outputs: image_id,dataset,scene,image,...
    """
    df = pd.read_csv(pred_csv)
    cols = set(df.columns)
    if "dataset" not in cols:
        raise ValueError("Predictions CSV must contain a `dataset` column")

    cluster_col = None
    for c in ["scene", "cluster", "cluster_label"]:
        if c in cols:
            cluster_col = c
            break
    if cluster_col is None:
        raise ValueError("Predictions CSV must contain one of: scene / cluster / cluster_label")

    image_col = "image" if "image" in cols else None
    image_id_col = "image_id" if "image_id" in cols else None
    if image_col is None and image_id_col is None:
        raise ValueError("Predictions CSV must contain `image` or `image_id`")

    df = df[["dataset", cluster_col] + ([image_col] if image_col else []) + ([image_id_col] if image_id_col else [])].copy()
    df = df.rename(columns={cluster_col: "pred_cluster"})

    def _row_to_key(r) -> str:
        return normalize_image_key(
            str(r["dataset"]),
            image_id=str(r[image_id_col]) if image_id_col and pd.notna(r[image_id_col]) else None,
            image=str(r[image_col]) if image_col and pd.notna(r[image_col]) else None,
        )

    df["image_key"] = df.apply(_row_to_key, axis=1)

    # Keep only images we actually expect (ignore licenses / stray rows).
    kept = []
    for dataset, g in df.groupby("dataset", sort=False):
        exp = expected_images.get(str(dataset), set())
        if exp:
            kept.append(g[g["image_key"].isin(exp)])
        else:
            kept.append(g)
    df = pd.concat(kept, ignore_index=True) if kept else df

    # If predictions do not cover all images, treat missing ones as predicted outliers.
    rows_missing = []
    for dataset, exp in expected_images.items():
        present = set(df.loc[df["dataset"] == dataset, "image_key"].tolist())
        missing = sorted(exp - present)
        if not missing:
            continue
        rows_missing.extend(
            {
                "dataset": dataset,
                "pred_cluster": outliers_label,
                "image_key": k,
            }
            for k in missing
        )
    if rows_missing:
        df = pd.concat([df, pd.DataFrame(rows_missing)], ignore_index=True)

    pred: Dict[str, Dict[str, set[str]]] = {}
    for (dataset, cluster), g in df.groupby(["dataset", "pred_cluster"], sort=True):
        pred.setdefault(str(dataset), {})[str(cluster)] = set(g["image_key"].tolist())

    return pred


def harmonic_mean(a: float, b: float) -> float:
    if a <= 0 or b <= 0:
        return 0.0
    return 2.0 / (1.0 / a + 1.0 / b)


@dataclass(frozen=True)
class DatasetScore:
    dataset: str
    mAA: float
    clustering_score: float
    final_score: float
    n_scenes: int
    n_images_in_scenes: int
    n_correct: int
    n_cluster_images_summed: int


def score_dataset(
    *,
    dataset: str,
    gt_scenes: Dict[str, set[str]],
    pred_clusters: Dict[str, set[str]],
    outliers_label: str = "outliers",
) -> Tuple[DatasetScore, pd.DataFrame]:
    candidates = {k: v for k, v in pred_clusters.items() if k != outliers_label and len(v) > 0}

    rows = []
    total_correct = 0
    total_scene_images = 0
    total_cluster_images = 0

    for scene, scene_imgs in sorted(gt_scenes.items()):
        n_scene = len(scene_imgs)
        total_scene_images += n_scene

        best_cluster = None
        best_maa = -1.0
        best_cs = -1.0
        best_correct = 0
        best_cluster_size = 0

        for c_label, c_imgs in candidates.items():
            inter = len(scene_imgs & c_imgs)
            maa = inter / n_scene if n_scene else 0.0
            cs = inter / len(c_imgs) if c_imgs else 0.0
            if (maa > best_maa) or (maa == best_maa and cs > best_cs):
                best_cluster = c_label
                best_maa = maa
                best_cs = cs
                best_correct = inter
                best_cluster_size = len(c_imgs)

        # If there are no candidate clusters, treat as fully missing.
        if best_cluster is None:
            best_cluster = "(none)"
            best_maa = 0.0
            best_cs = 0.0
            best_correct = 0
            best_cluster_size = 0

        total_correct += best_correct
        total_cluster_images += best_cluster_size

        rows.append(
            {
                "dataset": dataset,
                "gt_scene": scene,
                "assigned_cluster": best_cluster,
                "n_scene": n_scene,
                "n_cluster": best_cluster_size,
                "n_correct": best_correct,
                "mAA(scene)": best_maa,
                "CS(scene)": best_cs,
            }
        )

    maa = (total_correct / total_scene_images) if total_scene_images else 0.0
    cs = (total_correct / total_cluster_images) if total_cluster_images else 0.0
    final = harmonic_mean(maa, cs)

    ds_score = DatasetScore(
        dataset=dataset,
        mAA=float(maa),
        clustering_score=float(cs),
        final_score=float(final),
        n_scenes=len(gt_scenes),
        n_images_in_scenes=int(total_scene_images),
        n_correct=int(total_correct),
        n_cluster_images_summed=int(total_cluster_images),
    )

    return ds_score, pd.DataFrame(rows)


In [ ]:
# --- Configure paths ---

def find_repo_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "pyproject.toml").exists() or (p / "PIPELINE.md").exists():
            return p
    return start


REPO_ROOT = find_repo_root()
print("REPO_ROOT:", REPO_ROOT)

# Ground-truth labels are available for training data only.
GT_LABELS_CSV = REPO_ROOT / "data/train_labels.csv"
GT_DATA_ROOT = REPO_ROOT / "data/train"

# Predicted clustering labels (examples):
# - "cache/clusters.csv" or "cache/clusters_retrieval.csv" (dataset,image_id,scene)
# - a submission-like CSV with columns dataset,scene,image,...
PRED_CSV = REPO_ROOT / "cache/clusters_retrieval.csv"

# Label used to mark predicted outliers.
OUTLIERS_LABEL = "outliers"

if not GT_LABELS_CSV.exists():
    raise FileNotFoundError(
        f"Could not find GT labels at {GT_LABELS_CSV}. "
        "If you're running outside the repo, set GT_LABELS_CSV/GT_DATA_ROOT to absolute paths."
    )


REPO_ROOT: /home/shamika/documents/prog_workspace/ml_imc25/ImageMatchingChallenge2025


In [ ]:
# --- Load data ---
gt_scenes_by_dataset, all_images_by_dataset = load_ground_truth(labels_csv=GT_LABELS_CSV, data_root=GT_DATA_ROOT)
pred_clusters_by_dataset = load_predictions(pred_csv=PRED_CSV, expected_images=all_images_by_dataset, outliers_label=OUTLIERS_LABEL)

datasets = sorted(set(gt_scenes_by_dataset.keys()) & set(pred_clusters_by_dataset.keys()))
print(f"Datasets in GT: {len(gt_scenes_by_dataset)}")
print(f"Datasets in predictions: {len(pred_clusters_by_dataset)}")
print(f"Datasets scored: {len(datasets)} -> {datasets}")


Datasets in GT: 13
Datasets in predictions: 13
Datasets scored: 13 -> ['ETs', 'amy_gardens', 'fbk_vineyard', 'imc2023_haiper', 'imc2023_heritage', 'imc2023_theather_imc2024_church', 'imc2024_dioscuri_baalshamin', 'imc2024_lizard_pond', 'pt_brandenburg_british_buckingham', 'pt_piazzasanmarco_grandplace', 'pt_sacrecoeur_trevi_tajmahal', 'pt_stpeters_stpauls', 'stairs']


In [ ]:
# --- Score ---
dataset_scores = []
scene_tables = []

for ds in datasets:
    ds_score, per_scene = score_dataset(
        dataset=ds,
        gt_scenes=gt_scenes_by_dataset[ds],
        pred_clusters=pred_clusters_by_dataset[ds],
        outliers_label=OUTLIERS_LABEL,
    )
    dataset_scores.append(ds_score.__dict__)
    scene_tables.append(per_scene)

scores_df = pd.DataFrame(dataset_scores).sort_values("dataset")
scenes_df = pd.concat(scene_tables, ignore_index=True) if scene_tables else pd.DataFrame()

display(scores_df)

if len(scores_df) > 0:
    mean_final = float(scores_df["final_score"].mean())
    print(f"\nMean final score over datasets: {mean_final:.6f}")


,dataset,mAA,clustering_score,final_score,n_scenes,n_images_in_scenes,n_correct,n_cluster_images_summed
0,ETs,1.0,0.628571,0.77193,3,22,22,35
1,amy_gardens,0.0,0.000000,0.00000,1,200,0,0
2,fbk_vineyard,0.0,0.000000,0.00000,3,163,0,0
3,imc2023_haiper,0.0,0.000000,0.00000,3,54,0,0
4,imc2023_heritage,0.0,0.000000,0.00000,4,209,0,0
5,imc2023_theather_imc2024_church,0.0,0.000000,0.00000,2,76,0,0
6,imc2024_dioscuri_baalshamin,0.0,0.000000,0.00000,3,138,0,0
7,imc2024_lizard_pond,0.0,0.000000,0.00000,3,214,0,0
8,pt_brandenburg_british_buckingham,0.0,0.000000,0.00000,3,225,0,0
9,pt_piazzasanmarco_grandplace,0.0,0.000000,0.00000,2,168,0,0



Mean final score over datasets: 0.059379


In [ ]:
# --- Inspect per-scene assignment (optional) ---
# Show the best-matching predicted cluster for each GT scene.

if len(scenes_df) == 0:
    print("No per-scene results.")
else:
    display(
        scenes_df.sort_values(["dataset", "gt_scene"])[
            [
                "dataset",
                "gt_scene",
                "assigned_cluster",
                "n_scene",
                "n_cluster",
                "n_correct",
                "mAA(scene)",
                "CS(scene)",
            ]
        ]
    )


,dataset,gt_scene,assigned_cluster,n_scene,n_cluster,n_correct,mAA(scene),CS(scene)
0,ETs,ET,cluster_0002,9,9,9,1.0,1.000000
1,ETs,another_ET,cluster_0001,10,13,10,1.0,0.769231
2,ETs,outliers,cluster_0001,3,13,3,1.0,0.230769
3,amy_gardens,peach,(none),200,0,0,0.0,0.000000
4,fbk_vineyard,vineyard_split_1,(none),43,0,0,0.0,0.000000
5,fbk_vineyard,vineyard_split_2,(none),35,0,0,0.0,0.000000
6,fbk_vineyard,vineyard_split_3,(none),85,0,0,0.0,0.000000
7,imc2023_haiper,bike,(none),15,0,0,0.0,0.000000
8,imc2023_haiper,chairs,(none),16,0,0,0.0,0.000000
9,imc2023_haiper,fountain,(none),23,0,0,0.0,0.000000
